# Chapter 3: Dictionaries and Sets
## Fluent Python, 2nd Edition -- Distilled Interactive Notebook

> "Python is basically dicts wrapped in loads of syntactic sugar."
> -- *Lalo Martins*

Related wiki articles: [[dict-comprehensions-unpacking-merging]], [[pattern-matching-mappings]], [[hashability]], [[defaultdict-and-missing]], [[dict-variations]], [[immutable-mappings-and-views]], [[set-theory-and-operations]]

## TL;DR

- **Dict comprehensions** build dicts from iterables; `**` unpacks mappings; Python 3.9+ `|` and `|=` merge dicts.
- **Hashability** is the gatekeeper: dict keys and set elements must implement `__hash__` and `__eq__`; tuples are hashable only if all items are hashable.
- **defaultdict** and **`__missing__`** eliminate boilerplate for handling absent keys; **UserDict** is the safest base for custom mappings.
- **Dictionary views** (`.keys()`, `.items()`) are lightweight set-like projections -- no copying; `MappingProxyType` creates read-only proxies.
- **Sets** support mathematical operations (`|`, `&`, `-`, `^`) for union, intersection, difference, and symmetric difference -- far cleaner than loops.

---
## 1. Dict Comprehensions, Unpacking, and Merging

A **dictcomp** builds a dict from any iterable of key-value pairs, just like listcomps build lists. The `**` operator unpacks mappings. Python 3.9+ adds `|` and `|=` for merging.

See also: [[dict-comprehensions-unpacking-merging]]

In [ ]:
# Dict comprehension: build country->code mapping from tuples
dial_codes = [
    (880, 'Bangladesh'), (55, 'Brazil'), (86, 'China'),
    (91, 'India'), (62, 'Indonesia'), (81, 'Japan'),
    (234, 'Nigeria'), (92, 'Pakistan'), (7, 'Russia'),
    (1, 'United States'),
]
country_dial = {country: code for code, country in dial_codes}
print("country_dial:", country_dial)

# Filtered + transformed dictcomp
small_codes = {code: country.upper()
               for country, code in sorted(country_dial.items())
               if code < 70}
print("small codes:", small_codes)

In [ ]:
# ** unpacking in function calls and dict literals
def dump(**kwargs):
    return kwargs

result = dump(**{'x': 1}, y=2, **{'z': 3})
print("Unpacked into function:", result)

# ** in dict literals (later keys overwrite earlier)
merged = {'a': 0, **{'x': 1}, 'y': 2, **{'z': 3, 'x': 4}}
print("Merged literal:", merged)  # x=4 overwrites x=1

In [ ]:
# Merging with | and |= (Python 3.9+)
d1 = {'a': 1, 'b': 3}
d2 = {'a': 2, 'b': 4, 'c': 6}

# | creates a new dict (right side wins on conflicts)
d3 = d1 | d2
print("d1 | d2:", d3)
print("d1 unchanged:", d1)

# |= updates in place
d1 |= d2
print("d1 after |=:", d1)

---
## 2. Pattern Matching with Mappings (Python 3.10+)

`match/case` destructures mapping objects with **partial matching** -- extra keys are ignored. Capture remaining keys with `**extra`. Useful for semi-structured data like JSON.

See also: [[pattern-matching-mappings]]

In [ ]:
# Pattern matching on dict-like records
def get_creators(record: dict) -> list:
    match record:
        case {'type': 'book', 'api': 2, 'authors': [*names]}:
            return names
        case {'type': 'book', 'api': 1, 'author': name}:
            return [name]
        case {'type': 'book'}:
            raise ValueError(f"Invalid 'book' record: {record!r}")
        case {'type': 'movie', 'director': name}:
            return [name]
        case _:
            raise ValueError(f"Invalid record: {record!r}")

# Test cases
b1 = dict(api=1, author='Douglas Hofstadter', type='book', title='GEB')
print("b1 creators:", get_creators(b1))

b2 = dict(api=2, type='book', title='Python in a Nutshell',
          authors=['Martelli', 'Ravenscroft', 'Holden'])
print("b2 creators:", get_creators(b2))

m1 = {'type': 'movie', 'director': 'Kubrick', 'title': '2001'}
print("m1 creators:", get_creators(m1))

In [ ]:
# Capturing extra keys with **details
food = dict(category='ice cream', flavor='vanilla', cost=199)

match food:
    case {'category': 'ice cream', **details}:
        print(f"Ice cream details: {details}")

---
## 3. What Is Hashable

An object is **hashable** if it has a `__hash__()` that never changes during its lifetime and an `__eq__()` method. Dict keys and set elements **must** be hashable. Immutable built-ins (`str`, `bytes`, numerics) are always hashable. `tuple` is hashable only if all its items are hashable.

See also: [[hashability]]

In [ ]:
# Hashability demo
tt = (1, 2, (30, 40))       # tuple of immutables -> hashable
print(f"hash of {tt}: {hash(tt)}")

tf = (1, 2, frozenset([30, 40]))  # frozenset is hashable
print(f"hash of {tf}: {hash(tf)}")

tl = (1, 2, [30, 40])       # tuple containing list -> NOT hashable
try:
    hash(tl)
except TypeError as e:
    print(f"hash({tl}) -> TypeError: {e}")

In [ ]:
# User-defined objects are hashable by default (hash = id)
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

p = Point(1, 2)
print(f"hash(p) = {hash(p)}")
print(f"id(p)   = {id(p)}")

# Custom __eq__ without __hash__ makes instances unhashable
class BadPoint:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)

try:
    hash(BadPoint(1, 2))
except TypeError as e:
    print(f"BadPoint unhashable: {e}")

---
## 4. defaultdict and `__missing__`

- **defaultdict**: auto-creates missing values via a `default_factory` callable passed to the constructor.
- **`__missing__`**: dunder called by `dict.__getitem__` when a key is not found -- enables custom fallback logic.
- **UserDict**: the preferred base class for custom mappings (avoids pitfalls of subclassing `dict` directly).

See also: [[defaultdict-and-missing]]

In [ ]:
from collections import defaultdict

# defaultdict with list factory -- perfect for grouping
words = ['apple', 'banana', 'avocado', 'blueberry', 'cherry', 'apricot']
by_letter = defaultdict(list)
for word in words:
    by_letter[word[0]].append(word)

print("Grouped by first letter:")
for letter in sorted(by_letter):
    print(f"  {letter}: {by_letter[letter]}")

# Accessing missing key creates it
print("\nMissing key 'z':", by_letter['z'])  # creates empty list
print("'z' now in dict:", 'z' in by_letter)

In [ ]:
# setdefault: get-or-create in one lookup (no defaultdict needed)
index = {}
text = "to be or not to be that is the question"
for pos, word in enumerate(text.split()):
    index.setdefault(word, []).append(pos)

print("Word index:")
for word in sorted(index):
    print(f"  {word}: {index[word]}")

In [ ]:
# __missing__ dunder for custom key conversion
class StrKeyDict(dict):
    """Dict that converts non-string keys to str on lookup."""

    def __missing__(self, key):
        if isinstance(key, str):
            raise KeyError(key)  # prevent infinite recursion
        return self[str(key)]

    def __contains__(self, key):
        return key in self.keys() or str(key) in self.keys()

d = StrKeyDict([('2', 'two'), ('4', 'four')])
print("d['2']:", d['2'])     # normal string key
print("d[4]:",  d[4])        # int key -> converted to str
print("2 in d:", 2 in d)     # __contains__ checks str(key) too

---
## 5. Dict Variations: OrderedDict, ChainMap, Counter, UserDict

The `collections` module provides specialized dict subclasses for common patterns.

See also: [[dict-variations]]

In [ ]:
from collections import OrderedDict, ChainMap, Counter

# OrderedDict: order-sensitive equality
od1 = OrderedDict(a=1, b=2)
od2 = OrderedDict(b=2, a=1)
print("OrderedDict order matters:", od1 == od2)  # False

# Regular dict: order ignored for equality
d1 = dict(a=1, b=2)
d2 = dict(b=2, a=1)
print("dict order ignored:", d1 == d2)  # True

In [ ]:
# ChainMap: layered lookup (first map wins)
defaults = {'color': 'blue', 'size': 'medium'}
user_prefs = {'color': 'red'}

config = ChainMap(user_prefs, defaults)
print("color:", config['color'])  # 'red' from user_prefs
print("size:",  config['size'])   # 'medium' from defaults

# Writes go to the first map only
config['size'] = 'large'
print("user_prefs after update:", user_prefs)
print("defaults unchanged:", defaults)

In [ ]:
# Counter: multiset / tally
ct = Counter('abracadabra')
print("Letter counts:", ct)
print("Top 3:", ct.most_common(3))

# Arithmetic on Counters
ct.update('aaaaazzz')
print("After update:", ct)

# Counter subtraction
c1 = Counter(a=3, b=1)
c2 = Counter(a=1, b=3)
print("c1 - c2:", c1 - c2)   # drops zero/negative
print("c1 + c2:", c1 + c2)

---
## 6. Immutable Mappings and Dictionary Views

- **MappingProxyType**: wraps a dict in a read-only, dynamic proxy.
- **Dictionary views** (`.keys()`, `.values()`, `.items()`): lightweight, non-copying projections. `dict_keys` and `dict_items` support set operations.

See also: [[immutable-mappings-and-views]]

In [ ]:
from types import MappingProxyType

d = {1: 'A'}
d_proxy = MappingProxyType(d)

print("proxy[1]:", d_proxy[1])

# Can't modify through the proxy
try:
    d_proxy[2] = 'B'
except TypeError as e:
    print(f"Write blocked: {e}")

# But changes to the original dict are visible
d[2] = 'B'
print("proxy sees update:", d_proxy[2])

In [ ]:
# Dictionary views are dynamic and support set operations
d = dict(a=10, b=20, c=30)
keys = d.keys()
values = d.values()
items = d.items()

print("keys:", keys)
print("len:", len(keys))

# Views are dynamic -- they reflect mutations
d['z'] = 99
print("keys after adding 'z':", keys)

# dict_keys and dict_items support set operations
d1 = {'a': 1, 'b': 2, 'c': 3}
d2 = {'b': 20, 'c': 30, 'd': 40}
print("\nShared keys:", d1.keys() & d2.keys())
print("Keys in d1 only:", d1.keys() - d2.keys())
print("All keys:", d1.keys() | d2.keys())

---
## 7. Set Theory and Operations

`set` and `frozenset` enforce uniqueness via hash tables. They support mathematical operations: union (`|`), intersection (`&`), difference (`-`), and symmetric difference (`^`).

See also: [[set-theory-and-operations]]

In [ ]:
# Set basics: uniqueness and membership
l = ['spam', 'spam', 'eggs', 'spam', 'bacon', 'eggs']
unique = set(l)
print("Unique:", unique)

# Preserve order with dict.fromkeys
ordered_unique = list(dict.fromkeys(l))
print("Order-preserved unique:", ordered_unique)

# Set comprehension
from unicodedata import name
ascii_signs = {chr(i) for i in range(32, 128) if 'SIGN' in name(chr(i), '')}
print("ASCII signs:", ascii_signs)

In [ ]:
# Set operations
a = {1, 2, 3, 4, 5}
b = {4, 5, 6, 7, 8}

print("a | b  (union):       ", a | b)
print("a & b  (intersection):", a & b)
print("a - b  (difference):  ", a - b)
print("a ^ b  (sym diff):    ", a ^ b)

# Subset and superset
print("\n{1, 2} <= a:", {1, 2} <= a)   # subset
print("a >= {1, 2}:", a >= {1, 2})     # superset
print("{1, 2} < a:",  {1, 2} < a)      # proper subset

In [ ]:
# Practical: count needles in haystack using set intersection
haystack = set(range(10_000_000))
needles = {3, 7, 42, 999, 1_000_001}

found = needles & haystack
print(f"Found {len(found)} of {len(needles)} needles: {found}")

# frozenset: immutable and hashable -> can be a set element or dict key
fs = frozenset([1, 2, 3])
print(f"\nfrozenset hash: {hash(fs)}")
nested = {frozenset([1, 2]), frozenset([3, 4])}
print("Set of frozensets:", nested)

---
## Exercises

Test your understanding of Chapter 3 concepts.

### Exercise 1: Dict Comprehension
Given a list of words, create a dict mapping each word to its length using a dict comprehension.

In [ ]:
# Exercise 1: Your solution here
words = ['Python', 'is', 'fluent', 'and', 'powerful']

# Solution:
word_lengths = {word: len(word) for word in words}
print(word_lengths)

### Exercise 2: defaultdict Grouping
Use a `defaultdict` to group the numbers 1-20 by their remainder when divided by 5.

In [ ]:
# Exercise 2: Your solution here
from collections import defaultdict

# Solution:
by_remainder = defaultdict(list)
for n in range(1, 21):
    by_remainder[n % 5].append(n)

for rem in sorted(by_remainder):
    print(f"  remainder {rem}: {by_remainder[rem]}")

### Exercise 3: Set Operations
Given two sets of student names from different classes, find: (a) students in both, (b) students only in class A, (c) all students.

In [ ]:
# Exercise 3: Your solution here
class_a = {'Alice', 'Bob', 'Carol', 'Dave', 'Eve'}
class_b = {'Bob', 'Dave', 'Frank', 'Grace'}

# Solution:
print("In both classes:  ", class_a & class_b)
print("Only in class A:  ", class_a - class_b)
print("All students:     ", class_a | class_b)

---
## Key Takeaways

1. **Dict comprehensions** are the idiomatic way to build dicts from iterables. Use `**` for unpacking, `|` for merging (3.9+).
2. **Pattern matching** on mappings uses partial matching -- extra keys are silently ignored. Capture them with `**extra`.
3. **Hashability** requires both `__hash__` and `__eq__`. Mutable containers are not hashable; `frozenset` is the immutable set.
4. **defaultdict** and **setdefault** solve the "missing key" problem. For custom mappings, subclass **UserDict**, not dict.
5. **ChainMap** chains lookups across multiple dicts; **Counter** is a multiset that supports arithmetic.
6. **MappingProxyType** provides read-only dict views. Dictionary views (`.keys()`, `.items()`) support set operations.
7. **Set operations** (`|`, `&`, `-`, `^`) replace nested loops and conditionals with clean, declarative code.
8. **dict.fromkeys()** preserves insertion order while deduplicating -- a clean pattern for order-preserving unique.

See also: [[dict-comprehensions-unpacking-merging]], [[pattern-matching-mappings]], [[hashability]], [[defaultdict-and-missing]], [[dict-variations]], [[immutable-mappings-and-views]], [[set-theory-and-operations]]